# 00 — Data Checks
Description of all checks being performed per protocol §13.

In [ ]:
from pathlib import Path
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
import json

IS_KAGGLE = Path('/kaggle').exists()
EXP_LSH_ROOT = Path('/kaggle/input/exp-lsh-code/exp_lsh') if IS_KAGGLE else Path('..').resolve()
sys.path.insert(0, str(EXP_LSH_ROOT.parent))
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else EXP_LSH_ROOT / 'results' / 'data_checks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from exp_lsh.config.constants import SPLIT_SEEDS
except ImportError:
    SPLIT_SEEDS = [42, 43, 44, 45, 46]


In [ ]:
HEART_CSV_PATH = str(EXP_LSH_ROOT / 'data' / 'heart_disease' / 'heart.csv')
CHEST_EMBEDDINGS_PATH = str(EXP_LSH_ROOT / 'data' / 'chest_xray14' / 'embeddings.npy')
CHEST_LABELS_PATH = str(EXP_LSH_ROOT / 'data' / 'chest_xray14' / 'labels.npy')


In [ ]:
def check_no_nan_inf(X, name):
    if np.isnan(X).any() or np.isinf(X).any():
        return False, f"{name} contains NaN or Inf."
    return True, f"{name} is clean (no NaN/Inf)."

def check_no_overlap(idx_a, idx_b, name_a, name_b):
    overlap = set(idx_a).intersection(set(idx_b))
    if len(overlap) > 0:
        return False, f"{name_a} and {name_b} have {len(overlap)} overlapping samples!"
    return True, f"No overlap between {name_a} and {name_b}."

def check_sorted_by_distance(indices, distances):
    if not np.all(np.diff(distances, axis=1) >= 0):
        return False, "Distances are not sorted in ascending order."
    return True, "Distances are sorted correctly."


In [ ]:
try:
    heart_df = pd.read_csv(HEART_CSV_PATH)
    n_total_heart_orig = len(heart_df)
    heart_df = heart_df.replace([np.inf, -np.inf], np.nan).dropna()
    n_total_heart = len(heart_df)
    n_missing_dropped = n_total_heart_orig - n_total_heart
    
    print(f"Heart Disease dataset shape: {heart_df.shape}")
    print("Class distribution:\n", heart_df.iloc[:, -1].value_counts(normalize=True))
    
    ok, msg = check_no_nan_inf(heart_df.values, "Heart Disease DataFrame")
    print(msg)
except FileNotFoundError:
    print(f"Warning: {HEART_CSV_PATH} not found.")
    heart_df = None


In [ ]:
split_profiles = []
failed_checks = []

if heart_df is not None:
    X_heart = heart_df.iloc[:, :-1].values
    y_heart = heart_df.iloc[:, -1].values
    
    for seed in SPLIT_SEEDS:
        X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
            X_heart, y_heart, np.arange(len(X_heart)), test_size=0.20, random_state=seed, stratify=y_heart
        )
        X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
            X_temp, y_temp, idx_temp, test_size=0.20, random_state=seed, stratify=y_temp
        )
        
        ok_overlap1, msg1 = check_no_overlap(idx_train, idx_test, "Train", "Test")
        ok_overlap2, msg2 = check_no_overlap(idx_train, idx_val, "Train", "Val")
        ok_overlap3, msg3 = check_no_overlap(idx_val, idx_test, "Val", "Test")
        overlap_ok = ok_overlap1 and ok_overlap2 and ok_overlap3
        
        ok_nan_tr, _ = check_no_nan_inf(X_train, "Train")
        ok_nan_va, _ = check_no_nan_inf(X_val, "Val")
        ok_nan_te, _ = check_no_nan_inf(X_test, "Test")
        nan_ok = ok_nan_tr and ok_nan_va and ok_nan_te
        
        if not overlap_ok: failed_checks.append(f"Heart Seed {seed} overlap")
        if not nan_ok: failed_checks.append(f"Heart Seed {seed} nan")
        
        split_profiles.append({
            'dataset': 'Heart Disease',
            'split_seed': seed,
            'n_train': len(X_train),
            'n_val': len(X_val),
            'n_test': len(X_test),
            'overlap_ok': overlap_ok,
            'nan_ok': nan_ok
        })
        
        if seed == SPLIT_SEEDS[0]:
            X_train_heart_check = X_train
            X_val_heart_check = X_val
            X_test_heart_check = X_test


In [ ]:
if heart_df is not None:
    scaler = StandardScaler()
    scaler.fit(X_train_heart_check)
    X_val_scaled = scaler.transform(X_val_heart_check)
    X_test_scaled = scaler.transform(X_test_heart_check)
    
    ok_val, msg_val = check_no_nan_inf(X_val_scaled, "Scaled Val")
    ok_test, msg_test = check_no_nan_inf(X_test_scaled, "Scaled Test")
    print("StandardScaler checks:", msg_val, "|", msg_test)
    if not (ok_val and ok_test):
        failed_checks.append("Heart Scaler NaN")


In [ ]:
try:
    chest_embeddings = np.load(CHEST_EMBEDDINGS_PATH)
    chest_labels = np.load(CHEST_LABELS_PATH)
    print(f"ChestX-ray14 embeddings shape: {chest_embeddings.shape}")
    print(f"ChestX-ray14 labels shape: {chest_labels.shape}")
    
    if chest_embeddings.shape[1] != 1024:
        failed_checks.append(f"ChestX-ray14 expected d=1024, got {chest_embeddings.shape[1]}")
        
    ok_chest, msg_chest = check_no_nan_inf(chest_embeddings, "ChestX-ray14 Embeddings")
    print(msg_chest)
    if not ok_chest: failed_checks.append("ChestX-ray14 Embeddings NaN")
    
    unique, counts = np.unique(chest_labels, return_counts=True)
    print("Class distribution:\n", dict(zip(unique, counts / len(chest_labels))))
except FileNotFoundError:
    print(f"Warning: {CHEST_EMBEDDINGS_PATH} or {CHEST_LABELS_PATH} not found. Skipping ChestX-ray14.")
    chest_embeddings = None
    chest_labels = None


In [ ]:
if chest_embeddings is not None:
    for seed in SPLIT_SEEDS:
        X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
            chest_embeddings, chest_labels, np.arange(len(chest_embeddings)), 
            test_size=0.20, random_state=seed, stratify=chest_labels
        )
        X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
            X_temp, y_temp, idx_temp, test_size=0.20, random_state=seed, stratify=y_temp
        )
        
        ok_overlap1, _ = check_no_overlap(idx_train, idx_test, "Train", "Test")
        ok_overlap2, _ = check_no_overlap(idx_train, idx_val, "Train", "Val")
        ok_overlap3, _ = check_no_overlap(idx_val, idx_test, "Val", "Test")
        overlap_ok = ok_overlap1 and ok_overlap2 and ok_overlap3
        
        ok_nan_tr, _ = check_no_nan_inf(X_train, "Train")
        ok_nan_va, _ = check_no_nan_inf(X_val, "Val")
        ok_nan_te, _ = check_no_nan_inf(X_test, "Test")
        nan_ok = ok_nan_tr and ok_nan_va and ok_nan_te
        
        if not overlap_ok: failed_checks.append(f"Chest Seed {seed} overlap")
        if not nan_ok: failed_checks.append(f"Chest Seed {seed} nan")
        
        split_profiles.append({
            'dataset': 'ChestX-ray14',
            'split_seed': seed,
            'n_train': len(X_train),
            'n_val': len(X_val),
            'n_test': len(X_test),
            'overlap_ok': overlap_ok,
            'nan_ok': nan_ok
        })
        
        sizes_to_check = [10000, 50000]
        for sz in sizes_to_check:
            if len(X_train) >= sz:
                idx_subsample = idx_train[:sz]
                if len(idx_subsample) != sz:
                    failed_checks.append(f"Chest Seed {seed} subsample size mismatch for {sz}")
                ok_sub_query, _ = check_no_overlap(idx_subsample, idx_test, f"Base_{sz}", "Test_Query")
                if not ok_sub_query:
                    failed_checks.append(f"Chest Seed {seed} subsample overlap Query for {sz}")


In [ ]:
dataset_profiles = []
if heart_df is not None:
    dataset_profiles.append({
        'dataset': 'Heart Disease',
        'n_total': len(heart_df),
        'n_features': heart_df.shape[1] - 1,
        'n_classes': len(np.unique(heart_df.iloc[:, -1])),
        'n_missing_dropped': n_missing_dropped
    })
if chest_embeddings is not None:
    dataset_profiles.append({
        'dataset': 'ChestX-ray14',
        'n_total': len(chest_embeddings),
        'n_features': chest_embeddings.shape[1],
        'n_classes': len(np.unique(chest_labels)),
        'n_missing_dropped': 0
    })

dataset_profile = pd.DataFrame(dataset_profiles)
split_profile = pd.DataFrame(split_profiles)

print("=== Dataset Profile ===")
print(dataset_profile)
print("\n=== Split Profile ===")
print(split_profile)

dataset_profile.to_csv(OUTPUT_DIR / 'dataset_profile.csv', index=False)
split_profile.to_csv(OUTPUT_DIR / 'split_profile.csv', index=False)


In [ ]:
if len(failed_checks) == 0:
    print("\u2705 All checks passed")
else:
    print("\u274c Failed checks:")
    for fail in failed_checks:
        print(" -", fail)
